# Phase 3 — Feature Engineering

In [3]:
import pandas as pd
import numpy as np

sm = pd.read_csv("../data/cleaned/social_media_cleaned.csv")
teen = pd.read_csv("../data/cleaned/teen_mental_health_cleaned.csv")

In [4]:
sm.head()

,student_id,age,gender,academic_level,country,avg_daily_usage_hours,most_used_platform,affects_academic_performance,sleep_hours_per_night,mental_health_score,overall_impact
0,232,21,1,2,73,4,0,0,6.7,6.8,1
1,564,23,0,2,73,1,4,0,8.6,7.6,2
2,788,22,1,0,18,4,1,0,6.7,7.0,1
3,686,18,1,2,73,7,5,1,5.4,5.3,0
4,608,24,0,1,73,7,0,1,5.0,4.4,0


### Create Risk Score (Teen Dataset)

In [ ]:
teen['risk_score'] = (
    teen['stress_level'] * 0.3 +
    teen['anxiety_level'] * 0.3 +
    teen['addiction_level'] * 0.2 +
    teen['daily_social_media_hours'] * 0.1 +
    (10 - teen['sleep_hours']) * 0.1
)
# Helps in:
# Classification (High / Medium / Low risk)

```json
Meaning of Each Term
High Impact Factors (0.3 weight)
stress_level * 0.3
anxiety_level * 0.3
    Most important contributors to risk

Medium Impact (0.2)
addiction_level * 0.2
    Moderate influence

Low Impact (0.1)
daily_social_media_hours * 0.1
(10 - sleep_hours) * 0.1
    Smaller contribution
```

### Create Addiction Index (Teen Dataset)

In [7]:
teen['addiction_index'] = (
    teen['daily_social_media_hours'] * 0.5 +
    teen['screen_time_before_sleep'] * 0.3 +
    teen['addiction_level'] * 0.2
)

```json
Meaning of Each Feature
1. Most Important (0.5)
daily_social_media_hours * 0.5

Main driver
More hours = higher addiction

2. Behavioral Habit (0.3)
screen_time_before_sleep * 0.3

Very important signal
Using phone before sleep = strong addiction indicator

3. Self-reported Level (0.2)
addiction_level * 0.2
```

In [8]:
# less than 6 hours = poor sleep
teen['poor_sleep'] = (teen['sleep_hours'] < 6).astype(int)

In [9]:
teen.head()

,age,gender,daily_social_media_hours,platform_usage,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label,risk_score,addiction_index,poor_sleep
0,14,1,7,1,7.4,2.9,3.01,1.5,1,2,2,1,0,2.36,4.57,0
1,19,0,1,2,8.0,2.9,3.22,0.8,0,8,1,10,0,5.00,3.37,0
2,17,0,1,1,7.6,0.5,3.92,0.0,0,2,4,2,0,2.54,1.05,0
3,15,1,7,2,6.9,1.6,3.48,0.8,2,1,7,9,0,5.21,5.78,0
4,15,0,4,0,4.9,3.0,2.37,1.4,2,3,5,2,0,3.71,3.30,1


In [10]:
# more than 5 hours = high usage
teen['high_usage'] = (teen['daily_social_media_hours'] > 5).astype(int)

In [11]:
# Academic Performance Category (SM Dataset)

sm['performance_category'] = pd.cut(
    sm['mental_health_score'],
    bins=[0, 4, 7, 10],
    labels=['Low', 'Medium', 'High']
)
sm['performance_category'] = sm['performance_category'].astype(str)

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
sm['performance_category'] = le.fit_transform(sm['performance_category'])

In [13]:
sm.tail()

,student_id,age,gender,academic_level,country,avg_daily_usage_hours,most_used_platform,affects_academic_performance,sleep_hours_per_night,mental_health_score,overall_impact,performance_category
1700,701,20,0,2,44,4,6,0,7.2,7.0,2,2
1701,702,23,1,0,83,6,1,1,5.9,4.0,0,1
1702,703,21,0,2,20,5,9,1,6.7,6.0,0,2
1703,704,24,1,0,46,4,7,0,7.5,8.0,2,0
1704,705,19,0,2,79,6,0,1,6.3,5.0,0,2


### Merge Both Datasets

In [14]:
# Keeping common useful columns and then merge
sm_subset = sm[['age', 'gender', 'avg_daily_usage_hours',
                'sleep_hours_per_night', 'mental_health_score',
                'overall_impact']].copy()

sm_subset.rename(columns={
    'avg_daily_usage_hours': 'daily_social_media_hours',
    'sleep_hours_per_night': 'sleep_hours'
}, inplace=True)

merged = pd.merge(teen, sm_subset, on=['age', 'gender', 'daily_social_media_hours', 'sleep_hours'], how='left')

In [16]:
merged

,age,gender,daily_social_media_hours,platform_usage,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label,risk_score,addiction_index,poor_sleep,high_usage,mental_health_score,overall_impact
0,14,1,7,1,7.4,2.9,3.01,1.5,1,2,2,1,0,2.36,4.57,0,1,NaN,NaN
1,19,0,1,2,8.0,2.9,3.22,0.8,0,8,1,10,0,5.00,3.37,0,0,NaN,NaN
2,17,0,1,1,7.6,0.5,3.92,0.0,0,2,4,2,0,2.54,1.05,0,0,NaN,NaN
3,15,1,7,2,6.9,1.6,3.48,0.8,2,1,7,9,0,5.21,5.78,0,1,NaN,NaN
4,15,0,4,0,4.9,3.0,2.37,1.4,2,3,5,2,0,3.71,3.30,1,0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1226,18,0,6,1,6.6,2.0,2.76,1.0,1,3,4,4,0,3.84,4.40,0,1,NaN,NaN
1227,16,1,2,0,8.0,1.9,2.12,0.4,0,7,4,4,0,4.50,2.37,0,0,NaN,NaN
1228,14,0,1,0,8.7,0.7,3.98,0.8,0,1,1,1,0,1.03,0.91,0,0,NaN,NaN
1229,15,1,3,0,8.5,2.1,3.19,0.6,0,7,9,9,0,7.05,3.93,0,0,NaN,NaN


In [19]:
for col in merged.select_dtypes(include='number').columns:
    merged[col] = merged[col].fillna(merged[col].median())

In [20]:
# For last 2 NaN columns — the merge didn't find common matches. Just drop them
merged.drop(columns=['mental_health_score', 'overall_impact'], inplace=True, errors='ignore')

In [21]:
merged.head()

,age,gender,daily_social_media_hours,platform_usage,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label,risk_score,addiction_index,poor_sleep,high_usage
0,14,1,7,1,7.4,2.9,3.01,1.5,1,2,2,1,0,2.36,4.57,0,1
1,19,0,1,2,8.0,2.9,3.22,0.8,0,8,1,10,0,5.00,3.37,0,0
2,17,0,1,1,7.6,0.5,3.92,0.0,0,2,4,2,0,2.54,1.05,0,0
3,15,1,7,2,6.9,1.6,3.48,0.8,2,1,7,9,0,5.21,5.78,0,1
4,15,0,4,0,4.9,3.0,2.37,1.4,2,3,5,2,0,3.71,3.30,1,0


In [22]:
merged.drop(columns=['platform_usage'], inplace=True, errors='ignore')
print(merged.shape)
print(merged.columns.tolist())

(1231, 16)
['age', 'gender', 'daily_social_media_hours', 'sleep_hours', 'screen_time_before_sleep', 'academic_performance', 'physical_activity', 'social_interaction_level', 'stress_level', 'anxiety_level', 'addiction_level', 'depression_label', 'risk_score', 'addiction_index', 'poor_sleep', 'high_usage']


In [24]:
merged['risk_score'] = merged['risk_score'].round(2)
merged['addiction_index'] = merged['addiction_index'].round(2)

In [26]:
merged.to_csv("../data/cleaned/merged_data.csv", index=False)
print("Done ", merged.shape)

Done  (1231, 16)
